# Notebook 05 - ML: Trend Projection and Clustering (M6)

Fits linear trend models, runs K-Means clustering, evaluates models.

Syllabus: Unit V (supervised + unsupervised ML, train/validate/test)

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.utils.helpers import setup_logging
setup_logging()
sns.set_theme(style="whitegrid")
FIGURES = "../reports/figures"
os.makedirs(FIGURES, exist_ok=True)
print("Environment ready.")
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score
from src.utils.constants import (FACTORS_TIDY, FACTORS_WIDE, PLACES_CSV, SCORES_WIDE, TRENDS_CSV, CLUSTERS_CSV, METRICS_JSON, MODELS_DIR, DATA_PROCESSED)
from src.scoring.scoring import load_config, compute_scores
from src.models.trend import run_all_trends
from src.models.clustering import run_clustering
tidy = pd.read_parquet(FACTORS_TIDY)
wide = pd.read_csv(FACTORS_WIDE)
places = pd.read_csv(PLACES_CSV)
config = load_config()
DATA_PROCESSED.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
balanced = {'air_quality':0.30,'safety':0.25,'health':0.25,'economy':0.20}
all_years = sorted(wide['year'].dropna().unique().astype(int).tolist())
print(f'Years: {all_years}')

## 1. Compute factor scores for all years

In [ ]:
score_rows = []
for yr in all_years:
    try:
        sc = compute_scores(wide, config, balanced, year=yr)
        sc['year'] = yr
        score_rows.append(sc)
    except Exception as e:
        print(f'Year {yr}: {e}')
scores_long = pd.concat(score_rows, ignore_index=True) if score_rows else pd.DataFrame()
print(f'Scores: {len(scores_long)} rows across {len(all_years)} years')

## 2. Build factor score long table for trends

In [ ]:
from src.utils.constants import COVID_YEAR
factor_score_long_rows = []
for yr in all_years:
    try:
        sc = compute_scores(wide, config, balanced, year=yr)
        for _, row in sc.iterrows():
            fcols = [c for c in sc.columns if c.endswith('_score')]
            for fc in fcols:
                fn = fc.replace('_score','')
                factor_score_long_rows.append({'place_id':row['place_id'],'year':yr,'factor':fn,'score':row[fc],'gap_flag':'covid_year' if yr==COVID_YEAR else '','series_break':False})
    except: pass
fslong = pd.DataFrame(factor_score_long_rows)
print(f'Factor score long: {len(fslong)} rows, factors: {sorted(fslong["factor"].unique())}')

## 3. Trend projection (M6.1)

In [ ]:
print('Running trend projection with COVID year...')
trends_df, projections_df = run_all_trends(fslong, config, include_covid=True)
print('Running trend projection without COVID year (sensitivity)...')
trends_nc, proj_nc = run_all_trends(fslong, config, include_covid=False)
print(f'Trends: {len(trends_df)} place-factor pairs')
print('Outlook distribution:')
print(trends_df['outlook'].value_counts())
display(trends_df.head(10))

In [ ]:
# Visualise trend for top 5 cities by air quality
city_air = trends_df[(trends_df['factor']=='air_quality') & (trends_df['place_id'].isin(places[places['place_type']=='city']['place_id']))].merge(places[['place_id','place_name']], on='place_id')
top5 = city_air.dropna(subset=['slope']).nsmallest(5, 'slope')
if len(top5)>0:
    fig, axes = plt.subplots(1, len(top5), figsize=(14, 4), sharey=False)
    if len(top5)==1: axes=[axes]
    for ax, (_, row) in zip(axes, top5.iterrows()):
        hist = fslong[(fslong['place_id']==row['place_id']) & (fslong['factor']=='air_quality')]
        proj = projections_df[(projections_df['place_id']==row['place_id']) & (projections_df['factor']=='air_quality')]
        ax.plot(hist['year'], hist['score'], 'o-', color='#3498db', label='Historical', markersize=5)
        if not proj.empty:
            ax.plot(proj['year'], proj['yhat'], '--', color='#e74c3c', label='Projected')
            ax.fill_between(proj['year'], proj['lower'], proj['upper'], alpha=0.2, color='#e74c3c')
        ax.set_title(f"{row['place_name']}\n({row['outlook']})", fontsize=9)
        ax.set_ylim(0, 100)
        ax.legend(fontsize=7)
    plt.suptitle('Air Quality: 10-Year Projection', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/05_trend_projections.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Multiple linear regression (M6.2)

Target: health score. Predictors: air + economy + region (one-hot). Avoids circular regression.

In [ ]:
latest_yr = scores_long['year'].max()
latest = scores_long[scores_long['year']==latest_yr].merge(places[['place_id','region','place_type']], on='place_id', how='left')
latest = latest[latest['place_type']=='state']
fpreds = [c for c in latest.columns if c.endswith('_score') and c!='health_score']
if 'health_score' in latest.columns and fpreds:
    y = pd.to_numeric(latest['health_score'], errors='coerce')
    X = latest[fpreds].apply(pd.to_numeric, errors='coerce')
    rdum = pd.get_dummies(latest['region'], prefix='r', drop_first=True)
    X = pd.concat([X, rdum], axis=1)
    mask = y.notna() & X.notna().all(axis=1)
    if mask.sum() >= 5:
        mlr = sm.OLS(y[mask], sm.add_constant(X[mask].astype(float))).fit()
        print(mlr.summary())
    else: print('Not enough data for MLR.')
else: print('health_score not found in scores. Run pipeline first.')

## 5. Train/Validate/Test split (M6.3)

Time-based split (never random). Air: train=2015-2018, val=2019, test=2023-2024.

In [ ]:
metrics_all = {}
for pid in fslong[fslong['factor']=='air_quality']['place_id'].unique()[:8]:
    sub = fslong[(fslong['place_id']==pid) & (fslong['factor']=='air_quality')].dropna(subset=['score']).sort_values('year')
    if len(sub)<5: continue
    train = sub[sub['year']<=2018]; val = sub[sub['year']==2019]; test = sub[sub['year']>=2023]
    if len(train)<3: continue
    X_tr = sm.add_constant(train['year'].values.astype(float))
    m = sm.OLS(train['score'].values, X_tr).fit()
    res = {'place_id': pid}
    if not val.empty:
        yv = m.predict(sm.add_constant(val['year'].values.astype(float)))
        res['val_rmse'] = float(mean_squared_error(val['score'].values, yv, squared=False))
    if not test.empty:
        yt = m.predict(sm.add_constant(test['year'].values.astype(float)))
        res['test_rmse'] = float(mean_squared_error(test['score'].values, yt, squared=False))
        nb_pred = np.full(len(test), float(train['score'].iloc[-1]))
        res['baseline_rmse'] = float(mean_squared_error(test['score'].values, nb_pred, squared=False))
    metrics_all[pid] = res
metrics_df_m = pd.DataFrame(metrics_all).T
print('Validation metrics:')
display(metrics_df_m.round(3))

## 6. K-Means clustering (M6.4)

In [ ]:
latest_scores = scores_long[scores_long['year']==latest_yr].copy()
fsc = [c for c in latest_scores.columns if c.endswith('_score') and c!='score']
if fsc and len(latest_scores)>=4:
    clusters_df, profiles_df, km_metrics = run_clustering(latest_scores[['place_id']+fsc], fsc)
    print('Cluster profiles:')
    display(profiles_df)
    cn = clusters_df.merge(places[['place_id','place_name','place_type']], on='place_id', how='left')
    display(cn.head(20))
else:
    print('Not enough data for clustering.')
    clusters_df = pd.DataFrame(); km_metrics = pd.DataFrame()

In [ ]:
if not km_metrics.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10,4))
    axes[0].plot(km_metrics['k'], km_metrics['inertia'], 'o-', color='#3498db')
    axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method')
    axes[1].plot(km_metrics['k'], km_metrics['silhouette'], 'o-', color='#e74c3c')
    axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('Silhouette Score')
    plt.suptitle('K-Means: Choosing k', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/05_kmeans_elbow.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Export results

In [ ]:
import json
if not trends_df.empty:
    trends_df['last_year'] = latest_yr
    trends_df.to_csv(TRENDS_CSV, index=False)
    print(f'Saved: {TRENDS_CSV}')
if not clusters_df.empty:
    clusters_df.to_csv(CLUSTERS_CSV, index=False)
    print(f'Saved: {CLUSTERS_CSV}')
metrics_out = {}
if not metrics_df_m.empty:
    metrics_out['air_quality'] = {'val_rmse': float(metrics_df_m['val_rmse'].mean()) if 'val_rmse' in metrics_df_m.columns else None, 'test_rmse': float(metrics_df_m['test_rmse'].mean()) if 'test_rmse' in metrics_df_m.columns else None, 'baseline_rmse': float(metrics_df_m['baseline_rmse'].mean()) if 'baseline_rmse' in metrics_df_m.columns else None}
with open(METRICS_JSON, 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f'Saved: {METRICS_JSON}')

## Summary

This notebook:
- Fitted OLS linear trends per (place, factor) with actual year values to handle gaps correctly.
- Classified trends as Improving / Declining / Stable or uncertain (slope p-value, alpha=0.05).
- Generated 10-year projections with 95% prediction intervals, clipped to [0, 100].
- Used time-based split (never random): train 2015-2018, val 2019, test 2023-2024.
- Compared against naive baseline (last observed value repeated).
- Clustered places into livability tiers with K-Means.

**Wide intervals are expected and honest** with only 5-10 data points per series.